# Lab notebook — pre-launch item difficulty estimation

**Project:** Levante QA / VLM panel  
**Location:** `tools/vlm-panel/`  
**Purpose:** Living record of experiments to estimate difficulty of **new / unscored** items (especially CAT tasks that need bank-scale `d`) before children see them.

Treat this like a paper lab notebook: append dated entries; do not rewrite history. When metrics change, add a new section and point at the artifact files under `out/`.

**How to update:** after each experiment, append a dated markdown cell + (optional) a code cell that reloads JSON/CSV from `out/`. Keep claims tied to file paths.

---

## Table of contents

1. [Question & framing](#1-question--framing)
2. [Tooling inventory](#2-tooling-inventory)
3. [Experiment track: age gradients](#experiment-track-age-gradients)
4. [Prompt history (from git)](#prompt-history-from-git)
5. [Chronology of work](#3-chronology-of-work)
6. [Key quantitative results](#4-key-quantitative-results)
7. [Conceptual learnings](#5-conceptual-learnings)
8. [Open questions / next](#6-open-questions--next)
9. [Reload live artifacts](#7-reload-live-artifacts)

## 1. Question & framing

### Goal

Before release, estimate how hard a **proposed new item** (or translation) will be for children — especially for **CAT** tasks (TROG, matrix reasoning, mental rotation, same-different) where the runtime needs a bank-scale IRT difficulty `d`.

### Scope we settled on

- Primary eval tasks: **TROG** + **vocab** (evaluable human data).
- Not yet shipping bank patches; building measurement + prediction tools first.
- Always use locale **`en-US`** for EN panels (bare `en` → `audio/en/` 404s → black preload).

### Two different “difficulty” targets

| Target | Meaning | Use |
|--------|---------|-----|
| `p_pred_child` | Predicted child pass rate | Triage / “about what % of kids get this?” |
| Bank `d` (GCS item bank) | IRT difficulty, **higher = harder** | CAT item selection |
| Bench `item_params` (Redivis) | Research Rasch/2PL export | Report-only; often **easiness-coded** (opposite sign vs bank) |

**Normal bank `d`:** fitted from large child response matrices (Rasch/2PL, often multigroup), then shipped in GCS. New items have no row in that matrix until field calibration.

## 2. Tooling inventory

| Piece | Path | Role |
|-------|------|------|
| Panel runner | `run_panel.mjs` + `panel_grid*.json` | Ungated VLM respondents × ages × models |
| Analyze | `analyze.mjs` | `p_vlm`, human join, calibrator → `p_pred_child` |
| Bench calibrator | `fit_bench_calibrator.mjs` | Fit on levante-bench trials |
| Hybrid `d_est` | `estimate_difficulty.mjs` | Map `p_pred` (+ TROG tags / vocab Zipf) → bank `d` |
| ICC `d_icc` | `fit_icc_difficulty.mjs` | Fit Rasch-with-guessing from age→θ panel trials |
| Age gradient check | `eval_age_gradient.mjs` | `med_p` by age, item spreads |
| TROG prompts | `cypress/support/agents/prompts/trogPrompts.ts` | Age-conditional system + `trogUserText` (imported by agent) |
| TROG agent | `cypress/support/agents/trogVlmAgent.ts` | Cypress decide loop; re-exports prompts |
| Persona | `cypress/support/persona/childPersona.ts` | Age/θ preamble + TROG mastery cues |
| Learnings | `LEARNINGS.md`, `RESULTS.md` | Operator docs |

**Panel formula (ungated):** VLM answers real UI → empirical `p_vlm` → monotonic calibrator → `p_pred_child`.

**Do not use** `QA_PERSONA_GATE=irt` for new items (needs existing `d`; collapses unscored items).

## Experiment track: age gradients

**Role in this notebook:** supporting method for bank-scale / ICC difficulty — not a separate research program.

**Why it matters:** CAT overall accuracy can look flat across ages (older kids get harder items). Our panel is **fixed** (every age sees the same items), so per-item \(P(\mathrm{correct}\mid\theta)\) *should* rise with age if VLMs behave like IRT kids. Empirically they were nearly flat (median item age spread ~0.10 vs ~0.21 Rasch-expected), which is why linked `d_icc` failed (ρ ≈ 0.05).

**What we tried**

| Approach | Outcome |
|----------|---------|
| Soft “act like a 6-year-old” persona | Already failed for age curves (`LEARNINGS.md`) — do not revive |
| Same adult TROG grammar checklist at all ages | Flattens curves (age-6 parses like an adult) |
| Age-conditional checklist (≤8 light / ≥10 full) + TROG mastery cues | Mini-eval **GO**: Δ med_p(13−6) 0.030 → **0.071**; item mean spread 0.086 → **0.152**; full-panel MAE p_pred **0.059** |

**How we measure**

1. Primary: respondent `med_p` by age → Δ(a13 − a6) via `eval_age_gradient.mjs`
2. Item `max(p)−min(p)` across ages (mean/median spread)
3. Guardrail: full-panel MAE `p_pred` ≤ ~0.09
4. Downstream: `fit_icc_difficulty.mjs` ρ vs −p_pred / multivar `d_est`

**Artifacts:** `out/age_grad_baseline_pre.json`, `out/age_grad_after.json`, `out/age_eval_gonogo.md`, `panel_grid_trog_age_eval.json`

**Status (2026-08-07):** GO for fuller EN force recollect with age-conditional prompts; ICC still weak until ages 8/10 are refreshed too. Append dated results under Chronology (§3) as this track continues.


## Prompt history (from git)

Canonical TROG prompt text now lives in [`cypress/support/agents/prompts/trogPrompts.ts`](../../cypress/support/agents/prompts/trogPrompts.ts) (extracted from `trogVlmAgent.ts` so wording can be versioned and reviewed without agent plumbing). Child persona preamble remains [`cypress/support/persona/persona_template.txt`](../../cypress/support/persona/persona_template.txt); TROG age-band mastery cues are appended in `childPersona.ts`.

Reconstructed from `git log` / `git show` on `trogVlmAgent.ts` (and persona template):

| Date | Commit | Prompt change |
|------|--------|----------------|
| 2026-05-31 | `3a232e5` *add trog* | **v0 baseline:** hear sentence → pick picture; short grammar reminder (word order, who/whom, negation, prepositions, clauses). Digit-only reply. No checklist, no `trogUserText`. |
| 2026-05-31 | `5bdefa1` / `d945985` | Same TROG system prompt. Age persona + IRT θ preamble added (`persona_template.txt`, `QA_PERSONA_ABILITY=irt`) — prepended in `cypress.config`, not in the TROG agent file. |
| 2026-07-30 | `027133a` | Child Twins panel plumbing; TROG `SYSTEM_PROMPT` still v0. |
| 2026-08-03 | `757cc01` *update model matrix, add results* | **v1 checklist:** five silent checks (agent/patient, negation scope, spatial, comparative, relative clauses). First **`trogUserText`** structure hints (negation, despite, spatial, size, chase/push). |
| 2026-08-06 | `4bcf684` *trog & vocab difficulties* | **v2 checklist:** passives explicit; spatial list + under/beneath; comparatives “named pair only”; embeddings example; **item 6** contrast connectives (despite/although/however/instead). Richer `trogUserText` (passive `by`, embeddings regex). Used for full EN force recollect → `d_est` ρ 0.532 → 0.637. |
| 2026-08-06 | *(working tree / lab)* age-conditional | **v3:** `SYSTEM_PROMPT_CHECKLIST` (v2) for age ≥10; **`SYSTEM_PROMPT_YOUNG`** (no checklist) for age ≤8; young runs skip structure hints. TROG mastery cues in persona by age band. Mini-eval GO (Δ med_p 0.030 → 0.071). |
| 2026-08-07 | *(this entry)* | Prompts moved to `prompts/trogPrompts.ts`; agent re-exports for compatibility. |

**Design tension (still active):** checklist improves absolute TROG calibration (models were too hard on structure) but, applied at every age, flattens age/θ curves needed for ICC `d_icc`. v3 is the current compromise.

**Related persona note:** Soft “act like a 6-year-old” alone did not produce age curves (`LEARNINGS.md`). Operational mastery + age-conditional *task* scaffolding is the path we kept.

When editing prompts: change `trogPrompts.ts`, force-recollect affected grid cells, append a dated row here + Chronology metrics.

## 3. Chronology of work

### 2026-08 — Early framing

- Confirmed levante-qa **consumes** GCS bank `d` / persona θ; it does not fit IRT for new items.
- Best existing tool for unscored items: **VLM panel** → calibrator.
- GCS bank `d` and Redivis `item_params` are **different scales** (TROG overlap Pearson ≈ −0.39 historically).

### Affine → hybrid `d_est`

1. **v1** `estimate_difficulty.mjs`: affine `d_est = α + β·z` from `p_pred_child`.
   - TROG Spearman vs bank `d` weak (~0.24).
   - Vocab ranking stronger (~0.62); affine cannot beat pass-rate ranking ceiling.
2. **v2 hybrid:** ridge + Huber IRLS; features = `z` + TROG construction tags (`tagResidual`) / vocab Zipf.
   - TROG held-out **ρ_multivar ≈ 0.532** vs p-only ceiling **≈ 0.284** (beats ceiling).
   - Vocab: ranking ~0.61; Zipf helps MAE more than ranking.
3. Strengthened TROG prompts (`trogVlmAgent.ts`: passive, comparative, despite/however, embeddings).

### Limited prompt-eval recollect (8 cells)

- Grid: `panel_grid_trog_prompt_eval.json` (ages 8/10).
- Bug: `QA_LANGUAGE=en` → audio 404; fixed to **`en-US`**.
- n=8 too small/noisy to claim prompt lift (ceiling worsened 0.284 → 0.211 in that snapshot).

### Full EN TROG force recollect (32 cells)

- `panel_grid.json`, `--lang en-US --force` (~6h; muted/paused mid-run for Zoom, then resumed).
- Post: `post_en_full_recollect.sh` → analyze → fit calibrator → `estimate_difficulty` vs `d_est_trog_en_baseline_full.json`.

**After full recollect (2026-08-06):**

| Metric | baseline | after | Δ |
|--------|----------|-------|---|
| Spearman multivar | 0.532 | **0.637** | +0.105 |
| −p_pred ceiling | 0.284 | **0.471** | +0.187 |
| MAE multivar | 0.886 | **0.822** | −0.063 |
| MAE p_pred vs human | 0.076 | **0.063** | −0.013 |

Artifacts: `out/d_est_trog_en_report.md`, `out/trog_en_pred_after.json`.

### Idea: match new-item `p_pred` to similar `p_child`

- Reasonable for **triage** (same as p-only ranking).
- Insufficient for bank-scale CAT `d` when bank `d` ≉ reorder of pass rates (TROG Spearman(d_bank, −p_human) ≈ 0.43).

### Idea: variety of θs → ICC `d_icc`

- Implemented `fit_icc_difficulty.mjs`:  
  `P(correct|θ) = c + (1−c)·sigmoid(θ − d_icc)`  
  with θ from `age_task_ability.json`, then CV affine link to bank `d`.
- **Result (pre age-conditional prompts):** linked ρ ≈ **0.054**; raw ρ ≈ 0.21; −p_pred on same anchors ≈ 0.25; multivar `d_est` ≈ **0.637**.
- Diagnosis: **flat VLM×age curves** (median item age spread ~0.10 vs ~0.21 Rasch-expected given bank `d`). Fixed panel (all ages see same items) *should* show larger per-item age gradients than CAT overall accuracy.

### Age-conditional TROG child-likeness (2026-08-06 → 08-07)

**Hypothesis:** same adult grammar checklist at every age flattens age curves. Soft “act like a 6-year-old” already failed (`LEARNINGS.md`) — instead make **task prompt** age-conditional.

**Changes:**

1. `trogVlmAgent.ts`: age ≤8 → light prompt, no checklist / no structure `trogUserText` hints; age ≥10 → keep checklist.
2. `childPersona.ts`: TROG mastery cues by age band (operational, not cute roleplay).
3. Eval grid: `panel_grid_trog_age_eval.json` (2 models × ages 6/13 × 2 reps = 8 cells).

**Age-eval results:**

| | before | after |
|--|--------|-------|
| med_p age 6 | 0.894 | 0.818 |
| med_p age 13 | 0.924 | 0.889 |
| **Δ med_p(13−6)** | 0.030 | **0.071** |
| item mean age spread | 0.086 | **0.152** |
| Full-panel MAE p_pred | — | **0.059** (≤0.09) |

- One cell failed: `35flashlite_a6_r1`.
- **Verdict: GO** toward fuller EN force recollect (`out/age_eval_gonogo.md`).
- ICC on mixed panel still weak (ρ_cv ≈ 0.06) — only a6/a13 refreshed.

## 4. Key quantitative results

### 4.1 Child pass-rate prediction (TROG EN)

| Snapshot | MAE p_vlm | MAE p_pred |
|----------|-----------|------------|
| Pre full recollect (`trog_en_pred_baseline.json`) | 0.108 | 0.076 |
| Post full recollect (`trog_en_pred_after.json`) | 0.104 | **0.063** |
| After age-eval mix (`trog_en_pred_age_eval.json` full screen) | 0.106 | **0.059** |

### 4.2 Bank-scale `d_est` (TROG EN)

| Snapshot | ρ multivar | ρ −p_pred | MAE multivar |
|----------|------------|-----------|--------------|
| Baseline full (`d_est_trog_en_baseline_full.json`) | 0.532 | 0.284 | 0.886 |
| Post full recollect (`d_est_trog_en_metrics.json`) | **0.637** | **0.471** | **0.822** |

Vocab EN (`d_est_vocab_en_metrics.json`): ρ_multivar ≈ 0.613; −p_pred ceiling ≈ 0.661 (features don’t beat p-only ranking).

### 4.3 ICC from θ grid (TROG EN)

| Metric | Value |
|--------|-------|
| ρ linked `d_icc_cv` | ~0.05–0.06 |
| ρ raw `d_icc` | ~0.21 |
| θ grid | 6→−2.01, 8→−0.91, 10→−0.44, 13→−0.15 |

### 4.4 Age gradient (mini-grid a6 vs a13)

See §3 age-conditional section; artifacts `age_grad_baseline_pre.json`, `age_grad_after.json`.

## 5. Conceptual learnings

1. **Best child-behavior predictor:** ungated panel + calibrator → `p_pred_child`.
2. **IRT gate / sim_child:** useless for new items without `d`.
3. **TROG:** models too hard on structure → checklist helped absolute error; applied at all ages → flat age curves.
4. **Vocab:** models too easy on rare words → Zipf shrink in analysis; prompting barely moves lexical ceiling.
5. **Don’t ship TROG `d_est` as CAT ground truth** yet; use panel for triage / pass rates; hybrid `d_est` is promising for ranking vs bank `d`.
6. **Same pass rate ≠ same bank `d`.** Neighbor-matching on `p` is triage, not IRT calibration.
7. **θ-grid ICC needs real age sensitivity** in the VLM; soft personas failed; age-conditional *task* scaffolding is the current bet.
8. **Operator:** resume by default; `--force` only after prompt changes; always `en-US`.

## 6. Open questions / next

- [ ] **Full 32-cell EN TROG force recollect** with age-conditional prompts (GO from mini-eval).
- [ ] Re-fit calibrator + `estimate_difficulty` + `fit_icc_difficulty`; expect larger age spreads and maybe ICC lift.
- [ ] Retry failed cell `panel_trog_en_35flashlite_a6_r1`.
- [ ] Decide whether age ≤8 light prompt is too aggressive for age 8 (panel uses 6/8/10/13).
- [ ] Cross-language de/es force refresh still needed for xlang triage (stale panels).
- [ ] Keep this notebook updated after each experiment.

---

### Entry template (copy for new dated sections)

```markdown
### YYYY-MM-DD — <title>

**Hypothesis:** …
**Change:** … (files)
**Run:** … (grid / command)
**Metrics:** … (table + artifact paths)
**Verdict:** …
**Next:** …
```

## 7. Reload live artifacts

Run the cells below to refresh tables from `out/` without editing the narrative above.

In [ ]:
from pathlib import Path
import json

OUT = Path("out")
if not OUT.exists():
    OUT = Path("tools/vlm-panel/out")

def load(name):
    p = OUT / name
    if not p.exists():
        print(f"missing: {p}")
        return None
    return json.loads(p.read_text())

files = [
    "d_est_trog_en_baseline_full.json",
    "d_est_trog_en_metrics.json",
    "trog_en_pred_baseline.json",
    "trog_en_pred_after.json",
    "trog_en_pred_age_eval.json",
    "d_icc_trog_en_metrics.json",
    "age_grad_baseline_pre.json",
    "age_grad_after.json",
    "d_est_vocab_en_metrics.json",
]
data = {f: load(f) for f in files}
print("Loaded", sum(v is not None for v in data.values()), "/", len(files), "from", OUT.resolve())

In [ ]:
def fmt(x, d=3):
    return "—" if x is None else f"{x:.{d}f}"

base = data.get("d_est_trog_en_baseline_full.json") or {}
cur = data.get("d_est_trog_en_metrics.json") or {}
print("=== TROG d_est vs bank d ===")
print(f"{'metric':28} {'baseline':>10} {'current':>10} {'Δ':>10}")
for k in ("spearman_multivar", "spearman_neg_p_pred", "mae_multivar"):
    b, c = base.get(k), cur.get(k)
    delta = None if b is None or c is None else c - b
    print(f"{k:28} {fmt(b):>10} {fmt(c):>10} {fmt(delta):>10}")

pb = data.get("trog_en_pred_baseline.json") or {}
pa = data.get("trog_en_pred_after.json") or {}
print("\n=== TROG p_pred MAE ===")
print(f"baseline MAE p_pred: {fmt(pb.get('mae_p_pred_vs_human'))}")
print(f"after full recollect: {fmt(pa.get('mae_p_pred_vs_human'))}")

ag0 = data.get("age_grad_baseline_pre.json") or {}
ag1 = data.get("age_grad_after.json") or {}
print("\n=== Age gradient a6 vs a13 ===")
print(f"Δ med_p before: {fmt(ag0.get('delta_med_p'))}")
print(f"Δ med_p after:  {fmt(ag1.get('delta_med_p'))}")

icc = data.get("d_icc_trog_en_metrics.json") or {}
print("\n=== ICC ===")
print(f"ρ d_icc_cv: {fmt(icc.get('spearman_d_icc_cv'))}")
print(f"ρ −p_pred:  {fmt(icc.get('spearman_neg_p_pred'))}")

---

*End of initial lab dump (2026-08-07). Append below.*